In [1]:
import tensorflow as tf
import numpy as np

from huggingface_hub import snapshot_download

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import Activation, Dropout, Flatten, Dense
from tensorflow.keras.preprocessing import image

In [2]:
!pip install -q huggingface_hub hf_xet

In [3]:
dataset_path = snapshot_download(
    repo_id="shravya11/car-dataset",
    repo_type="dataset",
    local_dir="/content/car_dataset",
    allow_patterns=[
        "train/Audi/**",
        "train/Hyundai Creta/**",
        "train/Mahindra Scorpio/**",
        "train/Rolls Royce/**",
        "train/Swift/**",
        "train/Tata Safari/**",
        "train/Toyota Innova/**",
        "test/Audi/**",
        "test/Hyundai Creta/**",
        "test/Mahindra Scorpio/**",
        "test/Rolls Royce/**",
        "test/Swift/**",
        "test/Tata Safari/**",
        "test/Toyota Innova/**"
    ]
)

print("Dataset Downloaded Successfully!")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4165 files:   0%|          | 0/4165 [00:00<?, ?it/s]

Dataset Downloaded Successfully!


In [4]:
train = tf.keras.utils.image_dataset_from_directory(
    "/content/car_dataset/train",
    image_size=(128,128),
    batch_size=32
)

test = tf.keras.utils.image_dataset_from_directory(
    "/content/car_dataset/test",
    image_size=(128,128),
    batch_size=32,
    shuffle=False
)

class_names = train.class_names

print(class_names)

Found 3352 files belonging to 7 classes.
Found 813 files belonging to 7 classes.
['Audi', 'Hyundai Creta', 'Mahindra Scorpio', 'Rolls Royce', 'Swift', 'Tata Safari', 'Toyota Innova']


In [5]:
train = train.map(lambda x,y: (x/255.0,y))
test = test.map(lambda x,y: (x/255.0,y))

In [6]:
model = Sequential([
    Conv2D(32,(3,3),activation="relu",input_shape=(128,128,3)),
    MaxPooling2D((2,2)),
    Conv2D(64,(3,3),activation="relu"),
    MaxPooling2D((2,2)),
    Conv2D(128,(3,3),activation="relu"),
    MaxPooling2D((2,2)),
    Flatten(),
    Dense(256,activation="relu"),
    Dropout(0.5),
    Dense(len(class_names),activation="softmax")
])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [45]:
f = model.fit(
    train,
    epochs=20,
    validation_data=test
)

Epoch 1/20
105/105 ━━━━━━━━━━━━━━━━━━━━ 124s 1s/step - accuracy: 0.9687 - loss: 0.0942 - val_accuracy: 0.7036 - val_loss: 1.4233
Epoch 2/20
105/105 ━━━━━━━━━━━━━━━━━━━━ 141s 1s/step - accuracy: 0.9732 - loss: 0.0751 - val_accuracy: 0.7146 - val_loss: 1.4341
Epoch 3/20
105/105 ━━━━━━━━━━━━━━━━━━━━ 143s 1s/step - accuracy: 0.9800 - loss: 0.0624 - val_accuracy: 0.7183 - val_loss: 1.4902
Epoch 4/20
105/105 ━━━━━━━━━━━━━━━━━━━━ 143s 1s/step - accuracy: 0.9734 - loss: 0.0791 - val_accuracy: 0.7343 - val_loss: 1.5197
Epoch 5/20
105/105 ━━━━━━━━━━━━━━━━━━━━ 125s 1s/step - accuracy: 0.9866 - loss: 0.0444 - val_accuracy: 0.7331 - val_loss: 1.6241
Epoch 6/20
105/105 ━━━━━━━━━━━━━━━━━━━━ 124s 1s/step - accuracy: 0.9812 - loss: 0.0503 - val_accuracy: 0.7306 - val_loss: 1.6354
Epoch 7/20
105/105 ━━━━━━━━━━━━━━━━━━━━ 120s 1s/step - accuracy: 0.9812 - loss: 0.0585 - val_accuracy: 0.7085 - val_loss: 1.3904
Epoch 8/20
105/105 ━━━━━━━━━━━━━━━━━━━━ 122s 1s/step - accuracy: 0.9863 - loss: 0.0469 - val_accu

In [51]:
img = image.load_img("safari_600x400.png",target_size=(128,128))
img_arr = image.img_to_array(img)/255.0
img_arr = tf.expand_dims(img_arr,axis=0)
p = model.predict(img_arr)
print("Predicted Car:",class_names[np.argmax(p)])

print("Confidence:",round(float(np.max(p))*100,2),"%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Predicted Car: Tata Safari
Confidence: 65.82 %
